In [27]:
import pandas as pd
import numpy as np

print("Projet WC2026 ML lancé 🚀")


Projet WC2026 ML lancé 🚀


In [28]:
import pandas as pd
import numpy as np

# Charger le dataset
df = pd.read_csv("../data/results.csv")

# Afficher les 5 premières lignes
df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [29]:
print(df.shape)

(49437, 9)


In [30]:
print(df.columns)

Index(['date', 'home_team', 'away_team', 'home_score', 'away_score',
       'tournament', 'city', 'country', 'neutral'],
      dtype='str')


In [31]:
df.sample(5)

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
5468,1962-04-18,Hungary,Uruguay,1.0,1.0,Friendly,Budapest,Hungary,False
39780,2016-06-04,Estonia,Latvia,0.0,0.0,Baltic Cup,Tallinn,Estonia,False
8700,1971-12-15,Thailand,Laos,3.0,0.0,Southeast Asian Peninsular Games,Malaysia,Malaysia,True
47952,2024-11-17,Finland,Greece,0.0,2.0,UEFA Nations League,Helsinki,Finland,False
42825,2019-09-05,South Korea,Georgia,2.0,2.0,Friendly,Istanbul,Turkey,True


In [32]:
df = df[[
    "date",
    "home_team",
    "away_team",
    "home_score",
    "away_score",
    "tournament"
]]

In [33]:
df.sample(5)

,date,home_team,away_team,home_score,away_score,tournament
45959,2023-03-27,Chile,Paraguay,3.0,2.0,Friendly
29427,2005-08-17,Wales,Slovenia,0.0,0.0,Friendly
20075,1994-12-04,Ghana,Ivory Coast,1.0,1.0,Simba Tournament
10875,1977-02-28,Hong Kong,Indonesia,4.0,1.0,FIFA World Cup qualification
6442,1965-09-12,Zambia,Kenya,3.0,3.0,Friendly


In [34]:
def get_result(row):

    # Si l'équipe domicile marque plus
    if row["home_score"] > row["away_score"]:
        return 0

    # Si l'équipe extérieure marque plus
    elif row["home_score"] < row["away_score"]:
        return 2

    # Sinon match nul
    else:
        return 1

In [35]:
# Appliquer la fonction "get_result" à chaque ligne du DataFrame

In [36]:
df["result"] = df.apply(get_result, axis=1)

In [37]:
df.head()

,date,home_team,away_team,home_score,away_score,tournament,result
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,1
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,0
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,0
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,1
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,0


In [38]:
df["result"].value_counts()

result
0    24192
2    13948
1    11297
Name: count, dtype: int64

In [39]:
teams_stats = {}

In [40]:
for _, row in df.iterrows():

    home = row["home_team"]
    away = row["away_team"]

    if home not in teams_stats:
        teams_stats[home] = {
            "matches": 0,
            "wins": 0
        }

    if away not in teams_stats:
        teams_stats[away] = {
            "matches": 0,
            "wins": 0
        }

    teams_stats[home]["matches"] += 1
    teams_stats[away]["matches"] += 1

    if row["result"] == 0:
        teams_stats[home]["wins"] += 1

    elif row["result"] == 2:
        teams_stats[away]["wins"] += 1

In [41]:
teams_stats["France"]

{'matches': 937, 'wins': 476}

In [42]:
for team in teams_stats:

    matches = teams_stats[team]["matches"]
    wins = teams_stats[team]["wins"]

    teams_stats[team]["win_rate"] = wins / matches

In [43]:
teams_stats["France"]

{'matches': 937, 'wins': 476, 'win_rate': 0.5080042689434365}

In [44]:
def get_team_win_rate(team):
    return teams_stats[team]["win_rate"]

In [45]:
df["home_win_rate"] = df["home_team"].apply(get_team_win_rate)

df["away_win_rate"] = df["away_team"].apply(get_team_win_rate)

In [46]:
df[[
    "home_team",
    "away_team",
    "home_win_rate",
    "away_win_rate",
    "result"
]].head()

,home_team,away_team,home_win_rate,away_win_rate,result
0,Scotland,England,0.470726,0.571429,1
1,England,Scotland,0.571429,0.470726,0
2,Scotland,England,0.470726,0.571429,0
3,England,Scotland,0.571429,0.470726,1
4,Scotland,England,0.470726,0.571429,0


In [47]:
X = df[[
    "home_win_rate",
    "away_win_rate"
]]

y = df["result"]

In [48]:
X.head()

,home_win_rate,away_win_rate
0,0.470726,0.571429
1,0.571429,0.470726
2,0.470726,0.571429
3,0.571429,0.470726
4,0.470726,0.571429


In [49]:
y.head()

0    1
1    0
2    0
3    1
4    0
Name: result, dtype: int64

In [50]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [51]:
print(X_train.shape)
print(X_test.shape)

(39549, 2)
(9888, 2)


In [52]:
from sklearn.linear_model import LogisticRegression